# Trích xuất + dịch text game từ file IPA lớn — MeoRocky trên Google Colab

Notebook này chạy **trích xuất + dịch text** (bước "Trích xuất" và "Dịch" của tool BearTea) bằng
đúng code Python của repo [MunKai0408/MeoRocky](https://github.com/MunKai0408/MeoRocky),
nhưng chạy trên máy ảo Colab thay vì trong tab trình duyệt trên điện thoại.

**Dùng khi nào:** file IPA quá lớn cho Safari trên iPhone (tab chỉ dùng được
~1–1,5 GB RAM nên hay bị văng khi trích file >140 MB). Colab miễn phí có
~11 GB RAM dùng được nên trích được file IPA 2 GB trở lên.

**Cách chạy:** chỉ cần bấm **Run all** — notebook tự tìm file IPA trên Drive
của bạn rồi trích xuất và dịch. Kết quả (các file `.json` locale: `en.json`,
`zh-CN.json`, `zh-TW.json`… kèm file đã dịch `translated.vi.json`) được lưu
về Google Drive của bạn.

**Game mặc định:** Don't Starve Together 2.1.0 VH (file IPA ~1,69 GB đã có
trên Drive của bạn). Muốn trích game khác thì sửa 2 dòng `GAME_FILE_NAME` và
`DRIVE_FILE_ID` ở ô "Bước 1".

**Lưu ý:**
- Trích xuất đọc theo từng entry trong IPA (streaming), KHÔNG đọc cả file
  IPA vào RAM — RAM đỉnh phụ thuộc entry lớn nhất, không phải tổng dung
  lượng file. File 2 GB+ vẫn chạy được trên Colab miễn phí.
- Notebook trích xong thì **dịch luôn trên Colab** (Bước 3): mặc định dịch
  `en.json` → tiếng Việt, dùng Google Translate miễn phí (gtx, không cần
  key) hoặc worker đã deploy nếu bạn điền `WORKER_URL`. File đã dịch lưu về
  Drive dưới tên `translated.vi.json` — đúng định dạng tab Mod trên web nhận
  khi import, khỏi chuyển qua lại.
- Muốn chỉ trích xuất, không dịch: đặt `TRANSLATE_AFTER_EXTRACT = False`
  ở ô "Bước 1".
- File IPA phải nằm trên Google Drive của bạn trước (up lên Drive rồi làm
  theo bước 1 dưới đây).

> **Mới — nhận job từ web:** nếu đã tạo job trên web MeoRocky (tab Trích
> xuất > mục Colab), dán chuỗi job vào ô Job ngay sau ô clone repo (hoặc
> upload file `meorocky-job-<id>.json`), rồi bấm Run all — không cần gõ
> tay tên file/ID nữa. Muốn mở notebook là chạy luôn phần đầu: Colab >
> Notebook settings > Initialization > 'Run at startup' > 'First section'
> (cài 1 lần). Lưu ý: lần đầu Colab vẫn hiện bảng xin quyền truy cập
> Drive — bấm cho phép 1 lần, không tự động qua được bước này.


In [ ]:
# Gắn Google Drive vào máy ảo Colab (sẽ hiện bảng xin quyền truy cập Drive).
from google.colab import drive
drive.mount('/content/drive')
print('Đã gắn Drive tại /content/drive')

In [ ]:
# Clone repo MeoRocky (code trích xuất) về máy ảo + kiểm tra Python.
import os
import subprocess
import sys

REPO_DIR = '/content/MeoRocky'
if not os.path.isfile(os.path.join(REPO_DIR, 'pb3', 'bridge.py')):
    print('Đang clone repo MeoRocky...')
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/MunKai0408/MeoRocky.git', REPO_DIR],
                   check=True)
else:
    print('Repo đã có sẵn:', REPO_DIR)
print('Python:', sys.version.split()[0])
print('bridge.py:', os.path.isfile(os.path.join(REPO_DIR, 'pb3', 'bridge.py')))

In [ ]:
# Job handoff từ web MeoRocky (tùy chọn — bỏ qua nếu điền tay ở Bước 1).
# Web tạo job JSON (tab Trích xuất > mục Colab: nút 'Sao chép job' /
# 'Tải job.json'): dán chuỗi job vào JOB_JSON dưới đây, HOẶC upload file
# meorocky-job-<id>.json vào khung Files bên trái (đường dẫn /content/),
# HOẶC để web tự ghi job lên Drive (cần OAuth) rồi ô này đọc
# MeoRocky/jobs/latest.json. Không có job nào thì ô này không làm gì và
# Bước 1 dùng biến điền tay như cũ.
# Muốn mở notebook là chạy luôn phần đầu (gắn Drive + clone + job):
# Colab > Notebook settings > Initialization > 'Run at startup' >
# 'First section' (cài 1 lần trên tài khoản của bạn).
import os
import sys

JOB_JSON = ""  # dán chuỗi job JSON vào giữa "" (giữ nguyên "" nếu không dán)
JOB_UPLOAD_PATH = "/content/meorocky-job.json"
DRIVE_JOBS_DIR = "/content/drive/MyDrive/MeoRocky/jobs"
JOB_ID = ""
JOB_SOURCE = "none"

sys.path.insert(0, os.path.join(REPO_DIR, "pb3"))
from sdk.colab_job import resolve_job

try:
    _job, JOB_SOURCE = resolve_job(
        job_json_text=JOB_JSON,
        upload_path=JOB_UPLOAD_PATH,
        drive_jobs_dir=DRIVE_JOBS_DIR,
        job_id=JOB_ID or None)
except Exception as _e:  # noqa: BLE001 - job hỏng thì in rõ lý do rồi dừng
    print("Job lỗi: %s" % getattr(_e, "detail", _e))
    raise
if _job is not None:
    JOB = _job
    JOB_ID = _job["job_id"]
    GAME_FILE_NAME = _job["game_file_name"]
    DRIVE_FILE_ID = _job["drive_file_id"]
    FROM_LANG = _job["from_lang"]
    TO_LANG = _job["to_lang"]
    TRANSLATE_AFTER_EXTRACT = bool(_job["translate"])
    WORKER_URL = _job["worker_url"]
    print("Đã nhận job từ web (%s): %s — %s"
          % (JOB_SOURCE, _job["game"], _job["job_id"]))
    print("  Drive ID: %s | locale: %s | dịch %s->%s: %s"
          % (_job["drive_file_id"], ",".join(_job["locales"]),
             _job["from_lang"], _job["to_lang"],
             "có" if _job["translate"] else "không"))
else:
    JOB = None
    print("Không có job từ web — dùng biến điền tay ở Bước 1.")


## Bước 1 — Tự động tìm file IPA (điền game trước)

Điền game muốn trích ở đầu ô dưới: `GAME_FILE_NAME` (tên file `.ipa`
trên Drive) hoặc `DRIVE_FILE_ID` (ID lấy từ link Drive — đoạn giữa
`file/d/` và `/view`). Ô dưới tự làm 2 việc, theo thứ tự:

1. **Tìm theo tên** trong Drive đã gắn (`MyDrive` + Shared drives).
2. Nếu không thấy (vd file nằm ở "Shared with me", không có trong bản mount),
   **tự tải theo file ID** qua Drive API bằng tài khoản Google của bạn.

Chỉ khi cả 2 cách tự động đều thất bại mới cần điền tay vào
`MANUAL_IPA_PATH` (xem mục Lưu ý). Để trống cả hai thì ô này báo lỗi
và dừng lại.

Muốn đổi ngôn ngữ dịch: sửa `FROM_LANG` / `TO_LANG` ở đầu ô dưới
(mặc định en→vi). Nếu đã deploy worker dịch thì điền `WORKER_URL`
để dịch nhanh hơn gtx miễn phí.
Điền game trước khi chạy: `GAME_FILE_NAME` hoặc `DRIVE_FILE_ID` ở
đầu ô dưới (để trống cả hai thì ô này báo lỗi và dừng lại).


In [ ]:
# Bước 1 — TỰ ĐỘNG tìm/tải file IPA.
# TRƯỚC TIÊN điền game muốn trích ở dưới: GAME_FILE_NAME (tên file
# .ipa trên Drive, vd "9 Kings_1.0.6.ipa") HOẶC DRIVE_FILE_ID (ID lấy
# từ link Drive: mở file trên drive.google.com, copy đoạn giữa
# "file/d/" và "/view"). Điền 1 trong 2 là đủ; để trống cả hai thì
# ô này báo lỗi và dừng lại. Không còn game mặc định — game mặc định
# cũ (Don't Starve Together) không phải Unity nên chạy ra 0 file.
# --- Cấu hình dịch (Bước 3) ---
FROM_LANG = globals().get("FROM_LANG", "en")  # ngôn ngữ nguồn
TO_LANG = globals().get("TO_LANG", "vi")  # ngôn ngữ đích
TRANSLATE_AFTER_EXTRACT = globals().get("TRANSLATE_AFTER_EXTRACT", True)
WORKER_URL = globals().get("WORKER_URL", "")  # tùy chọn: URL worker đã deploy
# Để trống = dùng Google Translate miễn phí (gtx, không key, ~1 chuỗi/giây).
# Worker (Workers AI hoặc Google) dịch theo batch nên nhanh hơn nhiều.

import os
import subprocess

# Tôn trọng job từ cell job (nếu có): chỉ đặt mặc định khi chưa có biến.
GAME_FILE_NAME = globals().get("GAME_FILE_NAME", "")  # tên file .ipa trên Drive
DRIVE_FILE_ID = globals().get("DRIVE_FILE_ID", "")  # ID file trên Drive
SEARCH_ROOTS = ["/content/drive/MyDrive", "/content/drive/Shareddrives"]
DOWNLOAD_PATH = "/content/" + GAME_FILE_NAME
MANUAL_IPA_PATH = ""  # dự phòng cuối: dán đường dẫn mount vào đây nếu 2 cách tự động đều thất bại


def find_ipa_by_name(name, roots):
    """Tìm file theo tên trong Drive đã gắn. Trả đường dẫn đầu tiên tìm
    thấy, hoặc None. Dùng `find ... -print -quit` để dừng ngay khi thấy."""
    for root in roots:
        if not os.path.isdir(root):
            continue
        print("Đang tìm trong:", root)
        r = subprocess.run(["find", root, "-name", name, "-type", "f",
                            "-print", "-quit"],
                           capture_output=True, text=True)
        hits = [h for h in r.stdout.splitlines() if h.strip()]
        if hits:
            return hits[0]
    return None


def download_by_file_id(file_id, dest=None, service=None):
    """Tải file từ Drive theo file ID (Drive API, tài khoản Google của bạn
    trong Colab). Trả đường dẫn file đã tải, kiểm tra đủ dung lượng.
    dest để trống hoặc là thư mục thì lấy tên file từ Drive.
    `service` chỉ dùng để test (mặc định tự tạo từ tài khoản Colab)."""
    if service is None:
        from google.colab import auth
        auth.authenticate_user()
        from googleapiclient.discovery import build
        from google.auth import default
        creds, _ = default()
        service = build("drive", "v3", credentials=creds)
    from googleapiclient.http import MediaIoBaseDownload
    meta = service.files().get(fileId=file_id,
                               fields="name,size").execute()
    if not dest or dest.endswith("/"):
        dest = os.path.join(dest or "/content",
                            meta.get("name") or "game.ipa")
    expect = int(meta.get("size") or 0)
    print("Đang tải: %s (%.1f MB)" % (meta.get("name") or file_id,
                                      expect / 1e6 if expect else 0))
    request = service.files().get_media(fileId=file_id)
    with open(dest, "wb") as fh:
        dl = MediaIoBaseDownload(fh, request, chunksize=32 * 1024 * 1024)
        done = False
        while not done:
            status, done = dl.next_chunk()
            if status:
                print("  đã tải %.0f%%" % (status.progress() * 100), flush=True)
    got = os.path.getsize(dest)
    if expect and got != expect:
        raise IOError("Tải thiếu: được %d byte, cần %d byte" % (got, expect))
    print("Đã tải xong: %s (%.1f MB)" % (dest, got / 1e6))
    return dest


if MANUAL_IPA_PATH and os.path.isfile(MANUAL_IPA_PATH):
    IPA_PATH = MANUAL_IPA_PATH
    print("Dùng đường dẫn điền tay:", IPA_PATH)
elif not GAME_FILE_NAME and not DRIVE_FILE_ID:
    raise ValueError(
        "Chưa điền game nào: sửa GAME_FILE_NAME (tên file .ipa trên Drive) "
        "hoặc DRIVE_FILE_ID ở đầu ô này rồi chạy lại.\n"
        "Lấy ID: mở file trên drive.google.com, copy đoạn giữa 'file/d/' "
        "và '/view' trong đường link.")
else:
    IPA_PATH = (find_ipa_by_name(GAME_FILE_NAME, SEARCH_ROOTS)
                if GAME_FILE_NAME else None)
    if IPA_PATH:
        print("Đã tìm thấy file tại:", IPA_PATH)
    elif DRIVE_FILE_ID:
        print("Không tìm thấy bằng tên trong Drive đã gắn — "
              "chuyển sang tải theo file ID.")
        try:
            IPA_PATH = download_by_file_id(DRIVE_FILE_ID, DOWNLOAD_PATH)
        except Exception as e:  # noqa: BLE001 - rơi về hướng dẫn điền tay
            print("Tải theo file ID thất bại:", e)
            IPA_PATH = None
    else:
        print("Không tìm thấy %r trong Drive đã gắn." % GAME_FILE_NAME)
        IPA_PATH = None

if not IPA_PATH or not os.path.isfile(IPA_PATH):
    raise FileNotFoundError(
        "Không tự tìm/tải được file IPA.\n"
        "Cách dự phòng: mở khung Files bên trái → drive → MyDrive, tìm file .ipa, "
        "chuột phải → Copy path, dán vào MANUAL_IPA_PATH ở đầu ô này rồi chạy "
        "lại ô này.")
GAME_NAME = os.path.splitext(os.path.basename(IPA_PATH))[0]
OUT_DIR = "/content/drive/MyDrive/MeoRocky-output/%s" % GAME_NAME
print("IPA: %s (%.1f MB)" % (IPA_PATH, os.path.getsize(IPA_PATH) / 1e6))
print("Kết quả sẽ lưu về:", OUT_DIR)

## ⚠️ Lưu ý

- **Phiên Colab tối đa 12 tiếng; không động ~90 phút là tự ngắt; giờ cao điểm
  có thể bị ngắt ngẫu nhiên.** Trích 1 game chỉ mất vài phút–1 tiếng nên thời
  gian dư nhiều, nhưng đừng bấm chạy rồi bỏ đó quá lâu.
- **Kết quả PHẢI lưu về Drive ngay sau khi trích xong** — mọi file trên máy ảo
  Colab mất hết khi phiên tắt, chỉ còn thứ đã lưu về Drive.
- **Đừng tắt tab/đóng trình duyệt khi đang chạy.**
- **RAM dùng được ~11 GB** — notebook xử lý theo từng phần (streaming),
  KHÔNG đọc cả file IPA vào RAM, nên file 2 GB+ vẫn chạy được. Đừng tự ý sửa
  code thành đọc cả file vào RAM.
- **File IPA phải nằm sẵn trên Google Drive trước khi chạy** — đừng up trực
  tiếp file 2 GB từ iPhone lên Colab (lâu, dễ lỗi); up lên Drive trước rồi để
  Colab tải từ Drive.
- **Điền game trước khi chạy:** ô "Bước 1" để trống `GAME_FILE_NAME` và
  `DRIVE_FILE_ID` — bạn tự điền 1 trong 2: tên file `.ipa` trên Drive (vd
  `9 Kings_1.0.6.ipa`), hoặc ID lấy từ link Drive (mở file trên
  drive.google.com, copy đoạn giữa `file/d/` và `/view`). Điền tên thì
  notebook tìm file theo tên trong Drive đã gắn; không thấy thì tải theo ID
  (nếu có). Để trống cả hai thì ô Bước 1 báo lỗi và dừng lại.
- **Lần đầu chạy phải bấm cho phép Colab truy cập Drive.**
- **Nếu bị ngắt giữa chừng:** mở lại notebook chạy lại từ đầu là được (trích
  xuất nhanh); kết quả nào đã lưu về Drive thì vẫn còn nguyên.
- **Ví dụ: Don't Starve Together 2.1.0 VH đã thử trên bản mới
  nhất: trích ra 0 file — đây là kết quả ĐÚNG, không phải lỗi notebook.**
  File IPA này KHÔNG phải game Unity (không có `UnityFramework`,
  `data.unity3d`, `.assets` hay `global-metadata.dat` — kiểm tra thật trên
  file 1,69 GB: 8.654 entry, chỉ có dữ liệu engine riêng của Klei như
  `data/movies/*.ogv`, `data/sound/*.fsb`) nên bộ SDK Unity không tìm được
  text nào (275 mục bỏ qua; các file `.po` 17–19 MB bị trần plan 8 MiB
  chặn). Text thật của game nằm trong `data/databundles/scripts.zip`
  (4.067 entry: 4.024 file Lua + 11 file `.po` + `strings.pot`) — và file
  `vietnamese.po` (18.383.078 B) **đã dịch đầy đủ 100%**: 87.862 entry,
  87.829 `msgstr` có nội dung. Nghĩa là game đã có sẵn bản dịch Việt hoàn
  chỉnh trong file — chỉ cần bật tiếng Việt trong game, không cần dịch lại.
  Notebook vẫn đúng và có ích cho các game Unity lớn khác; riêng đường đọc
  `.po` của game Klei chưa có trong tool này.
- **Nếu notebook không tự tìm/tải được file:** mở khung Files bên trái →
  drive → MyDrive, tìm file `.ipa`, chuột phải → Copy path, dán vào
  `MANUAL_IPA_PATH` ở đầu ô "Bước 1" rồi chạy lại ô đó.
- **Dịch trên Colab (Bước 3):** mặc định dịch `en.json` → tiếng Việt ngay sau
  khi trích xong; file đã dịch `translated.vi.json` lưu về Drive cạnh file
  đã trích, đúng định dạng tab Mod trên web nhận khi import.
- **Tốc độ dịch:** gtx miễn phí ~1 chuỗi/giây (file 2252 key mất ~40 phút) —
  đừng tắt tab; nếu đã deploy worker thì điền `WORKER_URL` ở ô "Bước 1" để
  dịch theo batch, nhanh hơn nhiều. Hết hạn mức worker thì tự rơi về gtx.
- **Dịch lưu về Drive ngay sau mỗi file** — phiên ngắt giữa chừng thì file nào
  dịch xong vẫn còn; chạy lại ô Bước 3 để dịch tiếp (file nào đã đủ key thì
  bỏ qua).
- **Placeholder/thẻ không bao giờ bị dịch:** code dịch của repo che placeholder
  trước khi gửi và kiểm lại sau khi dịch (như trên web).


In [ ]:
# Bước 2 — Trích xuất (chạy đúng pipeline của tool web, locales en+zh, gộp 1 file/locale).
import hashlib
import json
import resource
import shutil
import sys
import time

t0 = time.monotonic()
sys.path.insert(0, os.path.join(REPO_DIR, 'pb3'))
import bridge

bridge.WEB_ROOT = os.path.join(REPO_DIR, 'pb3')
# Bản web giới hạn file ≤2 GB vì RAM tab trình duyệt; trên Colab không có
# giới hạn đó nên nới trần này (chỉ trong phiên Colab, không sửa repo).
bridge.MAX_WEB_INPUT = 64 * 1024 ** 3

boot = json.loads(bridge.boot())
print('Khởi động:', 'OK' if boot.get('ok') else 'LỖI', '| plugin:', boot.get('plugins'))
assert boot.get('ok'), boot

_last = {'t': 0.0, 'phase': ''}


def _progress(kind, done, total):
    now = time.monotonic()
    if kind != _last['phase'] or now - _last['t'] >= 2.0:
        _last['phase'] = kind
        _last['t'] = now
        if total and total > 0:
            print('  [%s] %d/%d (%.0f%%)' % (kind, done, total, 100.0 * done / total),
                  flush=True)
        else:
            print('  [%s] %d' % (kind, done), flush=True)


bridge.set_progress_cb(_progress)
res = json.loads(bridge.extract(IPA_PATH, locales=['en', 'zh'],
                                merge_one_file=True))
print('Trích xuất:', 'OK' if res.get('ok') else 'LỖI',
      '| quyết định:', res.get('decision'))
if not res.get('ok'):
    print('Lỗi:', res.get('error'))
    raise RuntimeError(res.get('error') or 'trích xuất thất bại')

# Copy file locale thật trong /tmp/pb3out về thư mục trên Drive.
os.makedirs(OUT_DIR, exist_ok=True)
for name in sorted(os.listdir('/tmp/pb3out')):
    src = os.path.join('/tmp/pb3out', name)
    if os.path.isfile(src):
        shutil.copy2(src, os.path.join(OUT_DIR, name))
# Ghi manifest handoff để web biết job này xong trích (web poll theo job_id).
try:
    from sdk.colab_job import write_extract_manifest
    _man = write_extract_manifest(
        OUT_DIR, res.get("merged_files") or [],
        job=globals().get("JOB"),
        game=GAME_NAME, drive_file_id=DRIVE_FILE_ID)
    print("Manifest handoff: %d file locale, job=%s"
          % (len(_man["files"]), _man["job_id"]))
except Exception as _e:  # noqa: BLE001 - manifest là phụ, không chặn trích xuất
    print("(không ghi được manifest handoff: %s)" % _e)


print()
print('=== Kết quả ===')
for name in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, name)
    kv = json.load(open(p, encoding='utf-8'))
    h = hashlib.md5(open(p, 'rb').read()).hexdigest()
    print('  %-12s %6d key  md5=%s' % (name, len(kv), h))
for m in res.get('merged_files') or []:
    print('  gộp: locale=%s file=%s keys=%d (trước gộp %d, xung đột %d)'
          % (m.get('locale'), m.get('file'), m.get('keys'),
             m.get('entries_before'), m.get('conflicts_total')))
print('Bỏ qua: %d mục (xem res["skipped"] nếu cần lý do từng mục)'
      % len(res.get('skipped') or []))
dt = time.monotonic() - t0
peak_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0
print('Thời gian: %.1f giây | RAM đỉnh: %.1f MB' % (dt, peak_mb))
print('File đã lưu về Drive tại:', OUT_DIR)

## Bước 3 — Dịch file locale vừa trích

Ô dưới dịch file `en.json` vừa trích sang tiếng Việt (mặc định), bằng đúng
code dịch của repo (`pb3/sdk/translate.py`): placeholder/thẻ được che trước
khi gửi và kiểm lại sau khi dịch, như trên web. Mặc định dùng Google
Translate miễn phí (gtx, không key, ~1 chuỗi/giây — file 2252 key mất
~40 phút); nếu đã deploy worker dịch thì điền `WORKER_URL` ở ô "Bước 1"
để dịch theo batch, nhanh hơn nhiều. File đã dịch lưu về Drive dưới tên
`translated.vi.json` — đúng định dạng tab Mod trên web nhận khi import.


In [ ]:
# Bước 3 — Dịch file locale vừa trích (mặc định en→vi).
# Dùng đúng code dịch của repo (pb3/sdk/translate.py): masking placeholder +
# verify+repair (PR #9) qua translate_json_file — file xuất được verify lại
# trước khi ghi. Mặc định dùng Google Translate miễn phí (gtx, không key,
# ~1 chuỗi/giây — file 2252 key mất ~40 phút); nếu WORKER_URL đã deploy
# (Workers AI hoặc Google) thì gọi worker (batch tới 100 chuỗi/request)
# nên nhanh hơn nhiều. Hết hạn mức worker thì tự rơi về gtx.
import hashlib
import json
import os
import sys
import time
import urllib.error
import urllib.request

sys.path.insert(0, os.path.join(REPO_DIR, 'pb3'))
from core.errors import PB3Error
from sdk.translate import (GoogleTranslateProvider, OfflineGlossaryProvider,
                           TranslationProvider, check_lang,
                           translate_json_file)


class WorkerProvider(TranslationProvider):
    """Gọi worker dịch BearTea (Cloudflare) đã deploy: POST {texts, target,
    source?} -> {ok, translated:[...]}. Batch tối đa 100 chuỗi/request, mỗi
    chuỗi ≤2000 ký tự (giới hạn worker); chuỗi dài hơn dịch bằng gtx (truyền
    vào). Hết hạn mức tháng (quota_exceeded) hoặc mạch ngắt (breaker_open)
    thì raise E_NETWORK để translate_table rơi về fallback (gtx)."""

    name = "worker"
    MAX_BATCH = 100
    MAX_CHARS = 2000

    def __init__(self, url, *, gtx=None, timeout=20.0, urlopen=None):
        self.base = url.rstrip("/")
        self.gtx = gtx
        self.timeout = timeout
        self._urlopen = urlopen or urllib.request.urlopen

    def healthy(self):
        try:
            req = urllib.request.Request(self.base + "/", method="GET")
            with self._urlopen(req, timeout=self.timeout) as resp:
                data = json.loads(resp.read().decode("utf-8"))
            return bool(isinstance(data, dict) and data.get("ok") and
                        data.get("service") == "beartea-translate-proxy")
        except Exception:
            return False

    def _post(self, payload):
        req = urllib.request.Request(
            self.base + "/",
            data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json"}, method="POST")
        try:
            with self._urlopen(req, timeout=self.timeout) as resp:
                return json.loads(resp.read().decode("utf-8"))
        except urllib.error.HTTPError as exc:
            try:
                return json.loads(exc.read().decode("utf-8"))
            except Exception:
                raise PB3Error("E_NETWORK", "worker trả HTTP %d" % exc.code)
        except (urllib.error.URLError, TimeoutError, OSError) as exc:
            raise PB3Error("E_NETWORK",
                           "không gọi được worker: %s" % type(exc).__name__)

    def translate_batch(self, texts, src, dst, on_item=None):
        out = [None] * len(texts)
        long_idx = [i for i, t in enumerate(texts) if len(t) > self.MAX_CHARS]
        short = [(i, t) for i, t in enumerate(texts) if len(t) <= self.MAX_CHARS]
        if long_idx and self.gtx is not None:
            got = self.gtx.translate_batch([texts[i] for i in long_idx],
                                           src, dst)
            for i, v in zip(long_idx, got):
                out[i] = v
        for start in range(0, len(short), self.MAX_BATCH):
            part = short[start:start + self.MAX_BATCH]
            data = self._post({"texts": [t for _, t in part],
                               "target": dst, "source": src})
            if not isinstance(data, dict) or not data.get("ok"):
                code = (data or {}).get("code") or ""
                if code in ("quota_exceeded", "breaker_open"):
                    raise PB3Error(
                        "E_NETWORK", "worker: %s" % code,
                        hint="Worker hết hạn mức hoặc đang ngắt mạch — rơi về gtx.")
                raise PB3Error(
                    "E_NETWORK",
                    "worker lỗi: %s" % ((data or {}).get("error") or code or "không rõ"))
            got = data.get("translated")
            if not isinstance(got, list) or len(got) != len(part):
                raise PB3Error("E_NETWORK",
                               "worker trả bản dịch thiếu/sai số lượng")
            for (i, _t), v in zip(part, got):
                out[i] = v if isinstance(v, str) else None
        if on_item is not None:
            for i in range(len(texts)):
                on_item(i)
        return out


def _translate_progress(done, total, state):
    now = time.monotonic()
    if done < total and now - state["t"] < 2.0:
        return
    state["t"] = now
    el = now - state["t0"]
    rate = done / el if el > 0 else 0.0
    eta = (total - done) / rate if rate > 0 else 0.0
    print("  dịch %d/%d (%.0f%%) — %.1f chuỗi/giây, còn ~%.0f phút"
          % (done, total, 100.0 * done / total if total else 100.0,
             rate, eta / 60.0), flush=True)


if not TRANSLATE_AFTER_EXTRACT:
    print("TRANSLATE_AFTER_EXTRACT=False — bỏ qua bước dịch.")
else:
    check_lang(FROM_LANG, allow_auto=False)
    check_lang(TO_LANG, allow_auto=False)
    if FROM_LANG == TO_LANG:
        raise ValueError("FROM_LANG và TO_LANG trùng nhau: %s" % TO_LANG)
    _src_name = {"en": "en.json", "zh": "zh-CN.json"}.get(
        FROM_LANG, FROM_LANG + ".json")
    _src_path = os.path.join(OUT_DIR, _src_name)
    _out_name = "translated.%s.json" % TO_LANG
    _out_path = os.path.join(OUT_DIR, _out_name)
    _skip = False
    if not os.path.isfile(_src_path):
        print("Không thấy file nguồn %s trong %s — bỏ qua bước dịch."
              % (_src_name, OUT_DIR))
        print("File có trong thư mục:",
              sorted(n for n in os.listdir(OUT_DIR) if n.endswith(".json")))
        _skip = True
    elif os.path.isfile(_out_path):
        try:
            _old = json.load(open(_out_path, encoding="utf-8"))
            _new = json.load(open(_src_path, encoding="utf-8"))
            _same = isinstance(_old, dict) and set(_old) == set(_new)
        except ValueError:
            _same = False
        if _same:
            print("Đã có %s đủ %d key — bỏ qua (xóa file để dịch lại)."
                  % (_out_name, len(_old)))
            _skip = True
    if not _skip:
        _gtx = GoogleTranslateProvider()
        if WORKER_URL:
            _worker = WorkerProvider(WORKER_URL, gtx=_gtx)
            if _worker.healthy():
                print("Worker đã deploy — dùng worker thay gtx.")
                _provider, _fb = _worker, _gtx
            else:
                print("WORKER_URL không trả lời đúng — rơi về gtx miễn phí.")
                _provider, _fb = _gtx, OfflineGlossaryProvider()
        else:
            _provider, _fb = _gtx, OfflineGlossaryProvider()
            _n = len(json.load(open(_src_path, encoding="utf-8")))
            print("Dùng gtx miễn phí (~1 chuỗi/giây): file %d key, "
                  "ước tính ~%.0f phút — đừng tắt tab." % (_n, _n / 60.0))
        _t0 = time.monotonic()
        _state = {"t": 0.0, "t0": _t0}
        try:
            _res = translate_json_file(
                _src_path, _out_path, source_lang=FROM_LANG,
                target_lang=TO_LANG, provider=_provider, fallback=_fb,
                progress_fn=lambda d, t: _translate_progress(d, t, _state),
                credit=True)
        except PB3Error as _exc:
            print("Dịch lỗi (%s): %s" % (_exc.code, _exc.message_vi))
            print("File nguồn vẫn còn nguyên; chạy lại ô này để thử lại.")
            raise
        _dt = time.monotonic() - _t0
        _md5 = hashlib.md5(open(_out_path, "rb").read()).hexdigest()
        _prov = {
            "game": GAME_NAME,
            "source_file": _src_name,
            "translationFile": _out_name,
            "from": FROM_LANG,
            "to": TO_LANG,
            "provider": _provider.name,
            "keys": int(_res.total),
            "untranslated": int(_res.untranslated),
            "glossary": int(_res.glossary),
            "cached": int(_res.cached),
            "credit_key": _res.credit_key,
            "seconds": round(_dt, 1),
            "md5": _md5,
            "note": "Dịch bằng notebook Colab MeoRocky.",
        }
        with open(os.path.join(OUT_DIR,
                               "translated.%s.provenance.json" % TO_LANG),
                  "w", encoding="utf-8") as _fh:
            json.dump(_prov, _fh, ensure_ascii=False, indent=2)
            _fh.write("\n")
            _fh.flush()
            os.fsync(_fh.fileno())
        print()
        print("=== Dịch xong ===")
        print("  nguồn: %s (%d key) -> %s" % (_src_name, _res.total, _out_name))
        print("  dịch bằng provider: %d | từ điển: %d | cache: %d | "
              "chưa dịch (giữ nguyên): %d"
              % (_res.done, _res.glossary, _res.cached, _res.untranslated))
        if _res.warnings:
            print("  cảnh báo (%d):" % len(_res.warnings))
            for _w in _res.warnings[:5]:
                print("   -", _w)
        print("  md5=%s | %.1f giây (%.2f key/giây)"
              % (_md5, _dt, _res.total / _dt if _dt else 0))
        print("  file đã lưu về Drive tại:", _out_path)
        # Cập nhật manifest handoff: dịch xong -> status 'done'.
        try:
            from sdk.colab_job import update_manifest_after_translate
            _man2 = update_manifest_after_translate(
                OUT_DIR, _out_path,
                job=globals().get("JOB"),
                game=GAME_NAME, drive_file_id=DRIVE_FILE_ID)
            print("  manifest: status=done, translated=%s" % _out_name)
        except Exception as _e:  # noqa: BLE001 - manifest là phụ, không chặn dịch
            print("  (không cập nhật được manifest handoff: %s)" % _e)


## Xong — nhớ kiểm tra trước khi đóng phiên

- **Kiểm tra file trên Drive:** mở thư mục `MeoRocky-output/<tên game>/` trên
  Google Drive, chắc chắn các file `.json` (đã trích + đã dịch) đã có đủ
  rồi mới đóng phiên.
- **Phiên Colab tối đa 12 tiếng** — trích 1 game thường chỉ mất vài phút tới
  ~1 tiếng nên dư thời gian. Nhưng **~90 phút không động vào notebook thì
  phiên tự ngắt**; file trên máy ảo Colab sẽ mất, chỉ còn lại những gì đã
  lưu về Drive. Đừng tắt tab khi đang chạy; nếu bị ngắt giữa chừng thì mở lại
  notebook chạy lại từ đầu là được, kết quả đã lưu về Drive vẫn còn nguyên.
- **Nếu trích ra 0 file hoặc ít key lạ** (vd chỉ vài chục key toàn chữ quảng cáo):
  game đó không có bảng ngôn ngữ có cấu trúc (text chính giấu ở chỗ khác) —
  dùng nút "Xem sâu" trên web để dò tiếp, hoặc báo lại để được hỗ trợ.
- **Bước tiếp theo:** file đã dịch `translated.vi.json` đã nằm trên Drive —
  import thẳng vào tab Mod trên web (nút "Ép lại vào file gốc") để ép vào
  file IPA, khỏi dịch lại trên web. Muốn dịch file khác thì sửa
  `FROM_LANG`/`TO_LANG` ở ô "Bước 1" rồi chạy lại ô "Bước 3".

Mở lại notebook này bất cứ lúc nào tại:
https://colab.research.google.com/github/MunKai0408/MeoRocky/blob/main/colab/extract-large-ipa.ipynb
